# 02 — Competition-faithful preprocessing

The original Challenge supplied labels only for **Set A**. Set B was a submission test set and Set C the final ranking set.

This notebook therefore fits every learned preprocessing quantity on **Set A only**. B/C are transformed with the frozen A state. Their outcome files are not read anywhere in this notebook.

For neural-model development, Set A is internally split into development-train and development-validation. This internal validation is our modern replacement for repeatedly probing held-out outcomes; after model choices are fixed, Notebook 03 retrains on all of A.


In [1]:
import sys, copy, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Start Jupyter from repository root or notebooks/.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.physionet import load_split
from src.data.preprocessing import Physionet2012Dataset, build_vocab

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
plt.rcParams["figure.dpi"] = 110
print("Project root:", PROJECT_ROOT)


Project root: C:\Users\guido\git_projects\icu-continuous-forecasting


In [2]:
A = load_split(RAW_DIR, split="set-a")
B = load_split(RAW_DIR, split="set-b")
assert len(A)==len(B)==4000
assert all(r.label is not None for r in A)

yA=np.asarray([int(r.label) for r in A])
idx=np.arange(len(A))
dev_train_idx, dev_val_idx=train_test_split(
    idx,test_size=.20,random_state=SEED,stratify=yA
)
A_train=[A[i] for i in dev_train_idx]
A_val=[A[i] for i in dev_val_idx]

display(pd.DataFrame([
    {"split":"A development train","n":len(A_train),"mortality":np.mean([r.label for r in A_train])},
    {"split":"A development validation","n":len(A_val),"mortality":np.mean([r.label for r in A_val])},
    {"split":"B challenge test","n":len(B),"mortality":np.nan},
]).round(4))


,split,n,mortality
0,A development train,3200,0.1384
1,A development validation,800,0.1388
2,B challenge test,4000,NaN


## 1. Development preprocessing: fit on A-development-train only

This state is used for architecture/training diagnostics on the internal A split.


In [3]:
dev_vocab=build_vocab(A_train)
dev_train_ds=Physionet2012Dataset(A_train,dev_vocab)
dev_train_ds.fit_normalizer()
assert hasattr(dev_train_ds,"records")

dev_val_ds=copy.deepcopy(dev_train_ds)
dev_val_ds.records=A_val

print("Development vocabulary:",len(dev_vocab))


Development vocabulary: 36


## 2. Final Challenge preprocessing: fit on all Set A

Once hyperparameters are fixed, the final submitted model is allowed to learn from all labelled training data A. Vocabulary and normalization are therefore refit on all A and then frozen for B/C.


In [4]:
final_vocab=build_vocab(A)
final_A_ds=Physionet2012Dataset(A,final_vocab)
final_A_ds.fit_normalizer()
assert hasattr(final_A_ds,"records")

final_B_ds=Physionet2012Dataset(B, final_vocab, normalizer=final_A_ds.normalizer)

unseen_B=sorted(set(build_vocab(B))-set(final_vocab))
print("Final vocabulary:",len(final_vocab))
print("B variables absent from A vocabulary:",unseen_B)
assert not unseen_B


Final vocabulary: 36
B variables absent from A vocabulary: []


## 3. Tensor contract

Inspect batches without touching B/C outcomes.


In [5]:
for name,ds in [("A",final_A_ds),("B",final_B_ds)]:
    b=ds.collate(list(range(16)))
    print(name,
          "times",tuple(b.times.shape),
          "values",tuple(b.values.shape),
          "mask",tuple(b.mask.shape),
          "delta_t",tuple(b.delta_t.shape),
          "static",tuple(b.static.shape))
    assert tuple(b.times.shape)==(49,)
    assert b.values.shape==b.mask.shape==b.delta_t.shape
    assert b.values.shape[-1]==len(final_vocab)
    m=b.mask.numpy().astype(bool)
    assert np.array_equal(np.isnan(b.values.numpy()),~m)
    assert np.isfinite(b.values.numpy()[m]).all()
    assert np.isfinite(b.delta_t.numpy()).all()
    assert np.isfinite(b.static.numpy()).all()


A times (49,) values (16, 49, 36) mask (16, 49, 36) delta_t (16, 49, 36) static (16, 5)
B times (49,) values (16, 49, 36) mask (16, 49, 36) delta_t (16, 49, 36) static (16, 5)


## 4. Raw irregularity vs hourly representation

The current project bins the first 48 hours onto `t=0,...,48`. This is deliberately retained for fair comparison between GRU/GRU-D and later continuous-time models; a non-binned ablation remains future work.


In [6]:
rec=A_train[0]
counts={v:len(rec.values.get(v,[])) for v in dev_vocab}
example_var=max(counts,key=counts.get)
j=dev_vocab.index(example_var)
raw=np.asarray([p[0] for p in rec.values[example_var]],dtype=float)

one=dev_train_ds.collate([0])
occupied=one.times.numpy()[one.mask[0,:,j].numpy().astype(bool)]

print("Record:",rec.record_id,"Variable:",example_var)
print("Raw timestamps:",raw[:20])
print("Occupied hourly bins:",occupied[:20])


Record: 137750 Variable: DiasABP
Raw timestamps: [ 0.1         0.35        0.6         0.85        0.93333333  1.6
  1.68333333  1.76666667  2.6         3.6         4.6         5.6
  6.6         7.6         8.6         9.6        10.6        11.6
 12.6        13.6       ]
Occupied hourly bins: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19.]


## 5. Final preprocessing contract

```text
DEVELOPMENT
Set A → internal 80/20 split
        ↓
fit preprocessing on A-dev-train
        ↓
choose training settings on A-dev-validation

FINAL CHALLENGE MODEL
all Set A labels
        ↓
refit vocabulary + normalization on all A
        ↓
train final model on all A
        ↓
freeze everything
        ↓
predict B and C record-by-record
```

`Outcomes-b.txt` is not read here, and Set C is not loaded at all: it remains untouched until final evaluation in Notebook 03.
